In [0]:
# --------------------------------------------------------------
# KPI run snapshot
# --------------------------------------------------------------
# Records this run's headline numbers to <env>_metrics.kpi_run_snapshot (one
# row per run, MERGEd on master_run_id), and exposes it to the business
# dashboards as a view in <env>_semantic_layer. The dashboards show each
# headline number for this run next to the previous run, so a viewer can tell
# new data from restated history. See superstore_kpi_snapshot.py.
#
# Runs last, after every KPI view: the snapshot must describe what the
# dashboards are about to show.
# --------------------------------------------------------------
import os, sys
from datetime import datetime, timezone

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")
master_run_id = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="master_run_id")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_metrics_schema, get_semantic_layer_schema
from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
from superstore_kpi_snapshot import measures_sql, record_snapshot, snapshot_row

catalog = get_catalog()
metrics_schema = get_metrics_schema()
semantic_layer_schema = get_semantic_layer_schema()
logger_kpi_snapshot = get_superstore_logger("superstore_kpi_run_snapshot")

# -----------------------------
# Dry run
# -----------------------------
# Writes, so it must not write during a dry run -- same rule as the KPI views.
if is_dry_run(dbutils):
    log_event(logger_kpi_snapshot, "INFO", "Dry run: skipping kpi_run_snapshot write",
              dry_run=True, layer="kpi snapshot")
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=kpi_run_snapshot (no data written)")

# -----------------------------
# Trigger type
# -----------------------------
# Passed from the job as {{job.trigger.type}} (periodic, one_time,
# run_job_task, ...), so the dashboard can say whether a run was scheduled.
try:
    trigger_type = dbutils.widgets.get("trigger_type")
except Exception:
    trigger_type = "unknown"

# -----------------------------
# Snapshot
# -----------------------------
measures = spark.sql(
    measures_sql(catalog, semantic_layer_schema, metrics_schema, master_run_id)
).collect()[0].asDict()
row = snapshot_row(env, master_run_id, trigger_type, measures, datetime.now(timezone.utc))
table = record_snapshot(spark, catalog, metrics_schema, semantic_layer_schema, row)

log_event(
    logger_kpi_snapshot, "INFO", "KPI run snapshot recorded",
    layer="kpi snapshot", table=table, master_run_id=master_run_id,
    trigger_type=row["trigger_type"], data_through=str(row["data_through"]),
    revenue_total=row["revenue_total"], customers_total=row["customers_total"],
)
# The exit string is the only output the Jobs API returns for a notebook task.
dbutils.notebook.exit(
    f"KPI_SNAPSHOT_RECORDED run={master_run_id} trigger={row['trigger_type']} "
    f"data_through={row['data_through']} revenue_total={row['revenue_total']} "
    f"customers_total={row['customers_total']}"
)
